# Hiyerarsik DDPM ile UI Layout + Gorsel Uretimi
**Adimlar:**
1. Drive'i bagla ve ZIP'i ac
2. RICO veri setini indir
3. Veri setini kontrol et
4. Modelleri egit (Layout DDPM + Image Diffusion)
5. Ornekleme ve degerlendirme

**Runtime > Change runtime type > GPU > A100 secin!**

In [ ]:
# --- HUCRE 1: Drive Baglantisi ve Kurulum ---
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, zipfile

# ZIP dosyasinin Drive'daki yolu
ZIP_PATH = "/content/drive/MyDrive/colab_projesi_v3.zip"

# Calisma dizini
WORK_DIR = "/content/ui_ddpm"

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(WORK_DIR)
    print(f"ZIP acildi: {WORK_DIR}")
    for f in z.namelist():
        print(f"  {f}")

os.chdir(WORK_DIR)
print(f"\nCalisma dizini: {os.getcwd()}")

In [ ]:
# --- HUCRE 2: Bagimliliklari Kur ---
!pip install -q torch torchvision tqdm matplotlib numpy datasets Pillow

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Bellek: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
# --- HUCRE 3: RICO Veri Setini Indir ---
# Ilk seferde ~5-15 dk surer. Sonraki seferlerde Drive'dan hizlica yuklenir.

import os, sys
sys.path.insert(0, WORK_DIR)

DRIVE_DATA_DIR = "/content/drive/MyDrive/rico_data"
data_dir = os.path.join(WORK_DIR, "data")
os.makedirs(data_dir, exist_ok=True)

rico_jsons_src = os.path.join(DRIVE_DATA_DIR, "rico_jsons")
rico_jsons_dst = os.path.join(data_dir, "rico_jsons")
rico_images_src = os.path.join(DRIVE_DATA_DIR, "rico_images")
rico_images_dst = os.path.join(data_dir, "rico_images")

if os.path.exists(rico_jsons_src) and not os.path.exists(rico_jsons_dst):
    print("Drive'dan RICO verileri baglaniyor (symlink)...")
    os.symlink(rico_jsons_src, rico_jsons_dst)
    print("Tamamlandi!")

if os.path.exists(rico_images_src) and not os.path.exists(rico_images_dst):
    os.symlink(rico_images_src, rico_images_dst)

if not os.path.exists(rico_jsons_dst) or len(os.listdir(rico_jsons_dst)) == 0:
    print("RICO veri seti indiriliyor (ilk sefer)...")
    !pip install -q datasets
    os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
    from ricodownloader import download_and_save_rico
    download_and_save_rico(
        output_folder=os.path.join(DRIVE_DATA_DIR, "rico_jsons"),
        image_folder=os.path.join(DRIVE_DATA_DIR, "rico_images"),
        max_samples=50000,
    )
    if not os.path.exists(rico_jsons_dst):
        os.symlink(os.path.join(DRIVE_DATA_DIR, "rico_jsons"), rico_jsons_dst)
    if not os.path.exists(rico_images_dst):
        os.symlink(os.path.join(DRIVE_DATA_DIR, "rico_images"), rico_images_dst)
    print("RICO veri seti Drive'a kaydedildi!")
else:
    total = 0
    for root, dirs, files in os.walk(rico_jsons_dst):
        total += len([f for f in files if f.endswith('.json')])
    print(f"RICO JSON dosyalari hazir: {total} adet")

In [ ]:
# --- HUCRE 4: Veri Setini Kontrol Et ---
os.chdir(WORK_DIR)
from data_preprocessing import load_rico_dataset

data = load_rico_dataset()
if data:
    print(f"\nToplam layout: {len(data)}")
    ex = data[0]
    print(f"Ornek - Panel sayisi: {len(ex['panels'])}")
    print(f"Ornek - Bilesen sayisi: {len(ex['components'])}")
    print(f"Ornek - Panel[0]: {ex['panels'][0]}")
    print(f"Ornek - Comp[0]: {ex['components'][0]}")
else:
    print("HATA: Veri yuklenemedi!")

In [ ]:
# --- HUCRE 5: Layout Modelleri Egitimi (Baseline + Hiyerarsik) ---
# A100'de ~1-3 saat surebilir (300 epoch). Early stopping varsa daha erken bitebilir.

os.chdir(WORK_DIR)
import torch
torch.cuda.empty_cache()

EPOCHS = 300
BATCH_SIZE = 64

!python main.py --mode train --model both --epochs $EPOCHS --batch-size $BATCH_SIZE

print("\nLayout model egitimi tamamlandi!")

In [ ]:
# --- HUCRE 6: Image Diffusion Egitimi (Asama 2) ---
# Layout modelleri egitildikten sonra calistirin. A100'de ~1-2 saat surebilir.

os.chdir(WORK_DIR)
torch.cuda.empty_cache()

!python main.py --mode train --model image --epochs $EPOCHS --batch-size $BATCH_SIZE

print("\nImage Diffusion egitimi tamamlandi!")

In [ ]:
# --- HUCRE 7: Ornekleme + Degerlendirme ---

os.chdir(WORK_DIR)
torch.cuda.empty_cache()

!python main.py --mode all --model all --epochs 0 --num-samples 100 --batch-size $BATCH_SIZE

print("\nOrnekleme ve degerlendirme tamamlandi!")

In [ ]:
# --- HUCRE 8: Sonuclari Goruntule ---
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage
import glob

print("=== Layout Ornekleri ===")
for img_path in sorted(glob.glob("outputs/figures/*.png")):
    print(f"\n{os.path.basename(img_path)}")
    display(IPImage(filename=img_path, width=800))

print("\n=== Uretilen UI Gorselleri ===")
ui_images = sorted(glob.glob("outputs/samples/generated_ui_*.png"))[:10]
if ui_images:
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    for idx, (ax, img_path) in enumerate(zip(axes.flatten(), ui_images)):
        img = plt.imread(img_path)
        ax.imshow(img)
        ax.set_title(f"#{idx}", fontsize=10)
        ax.axis('off')
    plt.suptitle("Uretilen UI Gorselleri (Image Diffusion)", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("Henuz uretilen gorsel yok. Once Hucre 6 ve 7'yi calistirin.")

In [ ]:
# --- HUCRE 9: Checkpoint'lari Drive'a Kaydet ---
import shutil

DRIVE_CKPT_DIR = "/content/drive/MyDrive/ui_ddpm_checkpoints"
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

ckpt_dir = os.path.join(WORK_DIR, "outputs", "checkpoints")
if os.path.exists(ckpt_dir):
    for f in os.listdir(ckpt_dir):
        src = os.path.join(ckpt_dir, f)
        dst = os.path.join(DRIVE_CKPT_DIR, f)
        shutil.copy2(src, dst)
        print(f"  Kaydedildi: {f}")
    print(f"\nCheckpoint'lar Drive'a kaydedildi: {DRIVE_CKPT_DIR}")
else:
    print("Checkpoint dizini bulunamadi.")

In [ ]:
# --- HUCRE 10: Uretilen Gorselleri Drive'a Kaydet ---
DRIVE_SAMPLES_DIR = "/content/drive/MyDrive/ui_ddpm_samples"
os.makedirs(DRIVE_SAMPLES_DIR, exist_ok=True)

for src_dir in [os.path.join(WORK_DIR, "outputs", "samples"),
                os.path.join(WORK_DIR, "outputs", "figures")]:
    if os.path.exists(src_dir):
        for f in os.listdir(src_dir):
            shutil.copy2(os.path.join(src_dir, f), os.path.join(DRIVE_SAMPLES_DIR, f))

print(f"Uretilen gorseller Drive'a kaydedildi: {DRIVE_SAMPLES_DIR}")